In [101]:
import pandas as pd
import json
import glob

In [102]:
results_dir = '../results/offset_0'

In [103]:
offset = "offset_14"
all_results = glob.glob(f'../results/{offset}/*.json')

table_data = []
for result in all_results:
    with open(result, 'r') as f:
        data = json.load(f)
        for k, v in data.items():
            if k in ['Accuracy', 'IoU', 'AP']:
                data[k] = v['mean']        
            if k == 'train_config':
                v = str(v)
                if '[' in v:
                    data[k] = 15
                elif '5' in v:
                    data[k] = 5
                else:
                    data[k] = 2
            if k == 'window_len':
                data['window_len'] = int(v)
        table_data.append(data)

In [104]:
df = pd.DataFrame(table_data)
df.head()

,AP,IoU,Accuracy,number_of_samples,depth_channel_representation_mode,window_len,train_config,multiple_digits,model_path
0,0.241378,0.011461,0.971889,1184,original,50,"{'epochs': 2, 'learning_rate': 0.001, 'layers'...",True,C:\Users\SID-DRW\CSE Master\FunMDL\dsait4205-e...
1,0.330166,0.124554,0.961878,1184,original,50,"{'epochs': 5, 'learning_rate': 0.001, 'layers'...",True,C:\Users\SID-DRW\CSE Master\FunMDL\dsait4205-e...
2,0.246006,0.014516,0.973926,1184,original,50,"{'epochs': 2, 'learning_rate': 0.001, 'layers'...",True,C:\Users\SID-DRW\CSE Master\FunMDL\dsait4205-e...
3,0.284611,0.064620,0.968135,1184,original,50,"{'epochs': 5, 'learning_rate': 0.001, 'layers'...",True,C:\Users\SID-DRW\CSE Master\FunMDL\dsait4205-e...
4,0.434966,0.265492,0.964299,1184,original,50,"[{'epochs': 5, 'learning_rate': 0.001, 'layers...",True,C:\Users\SID-DRW\CSE Master\FunMDL\dsait4205-e...
5,0.266245,0.016162,0.972014,1907,original,20,"{'epochs': 2, 'learning_rate': 0.001, 'layers'...",True,C:\Users\SID-DRW\CSE Master\FunMDL\dsait4205-e...
6,0.227397,0.016102,0.969451,1907,original,20,"{'epochs': 2, 'learning_rate': 0.001, 'layers'...",True,C:\Users\SID-DRW\CSE Master\FunMDL\dsait4205-e...
7,0.227932,0.024324,0.972459,1907,original,20,"{'epochs': 2, 'learning_rate': 0.001, 'layers'...",True,C:\Users\tfrva\OneDrive\Documenten\CSE Master\...
8,0.308895,0.064178,0.965869,1907,original,20,"{'epochs': 5, 'learning_rate': 0.001, 'layers'...",True,C:\Users\SID-DRW\CSE Master\FunMDL\dsait4205-e...
9,0.312806,0.102656,0.964666,1907,original,20,"{'epochs': 5, 'learning_rate': 0.001, 'layers'...",True,C:\Users\SID-DRW\CSE Master\FunMDL\dsait4205-e...


In [105]:
metrics = ['Accuracy', 'IoU', 'AP']
grouped = df.groupby(['window_len','train_config','multiple_digits','depth_channel_representation_mode'])
means = grouped[metrics].mean(numeric_only=True)
stds = grouped[metrics].std(numeric_only=True)
stds.rename(columns=lambda x: x + '_std', inplace=True)
counts = pd.DataFrame(grouped.size(), columns=['n'])

combined = pd.concat([means, stds, counts], axis=1)
for metric in metrics:
    combined[metric] = combined.apply(lambda x: f"{x[metric]:.4f} ± {x[f'{metric}_std']:.4f}", axis=1)
    combined.drop([f'{metric}_std'], axis=1, inplace=True)
combined.to_csv(f'results_{offset}.csv', index=True)
combined

Accuracy  \
window_len train_config                                       multiple_digits depth_channel_representation_mode                    
10         [{'epochs': 5, 'learning_rate': 0.001, 'layers'... True            original                           0.9568 ± 0.0013   
           {'epochs': 2, 'learning_rate': 0.001, 'layers':... True            original                           0.9704 ± 0.0015   
           {'epochs': 5, 'learning_rate': 0.001, 'layers':... True            original                           0.9599 ± 0.0036   
20         [{'epochs': 5, 'learning_rate': 0.001, 'layers'... True            original                           0.9602 ± 0.0015   
           {'epochs': 2, 'learning_rate': 0.001, 'layers':... True            original                           0.9706 ± 0.0027   
           {'epochs': 5, 'learning_rate': 0.001, 'layers':... True            original                           0.9639 ± 0.0030   
50         [{'epochs': 5, 'learning_rate': 0.001, 'layers'... True            original                           0.9614 ± 0.0017   
           {'epochs': 2, 'learning_rate': 0.001, 'layers':... True            original                           0.9730 ± 0.0020   
           {'epochs': 5, 'learning_rate': 0.001, 'layers':... True            original                           0.9649 ± 0.0035   

                                                                                                                             IoU  \
window_len train_config                                       multiple_digits depth_channel_representation_mode                    
10         [{'epochs': 5, 'learning_rate': 0.001, 'layers'... True            original                           0.2535 ± 0.0077   
           {'epochs': 2, 'learning_rate': 0.001, 'layers':... True            original                           0.0126 ± 0.0127   
           {'epochs': 5, 'learning_rate': 0.001, 'layers':... True            original                           0.1205 ± 0.0284   
20         [{'epochs': 5, 'learning_rate': 0.001, 'layers'... True            original                           0.2572 ± 0.0157   
           {'epochs': 2, 'learning_rate': 0.001, 'layers':... True            original                           0.0192 ± 0.0086   
           {'epochs': 5, 'learning_rate': 0.001, 'layers':... True            original                           0.1075 ± 0.0323   
50         [{'epochs': 5, 'learning_rate': 0.001, 'layers'... True            original                           0.2887 ± 0.0158   
           {'epochs': 2, 'learning_rate': 0.001, 'layers':... True            original                           0.0143 ± 0.0106   
           {'epochs': 5, 'learning_rate': 0.001, 'layers':... True            original                           0.0873 ± 0.0379   

                                                                                                                              AP  \
window_len train_config                                       multiple_digits depth_channel_representation_mode                    
10         [{'epochs': 5, 'learning_rate': 0.001, 'layers'... True            original                           0.4319 ± 0.0118   
           {'epochs': 2, 'learning_rate': 0.001, 'layers':... True            original                           0.2321 ± 0.0347   
           {'epochs': 5, 'learning_rate': 0.001, 'layers':... True            original                           0.3221 ± 0.0249   
20         [{'epochs': 5, 'learning_rate': 0.001, 'layers'... True            original                           0.4323 ± 0.0110   
           {'epochs': 2, 'learning_rate': 0.001, 'layers':... True            original                           0.2552 ± 0.0255   
           {'epochs': 5, 'learning_rate': 0.001, 'layers':... True            original                           0.3265 ± 0.0168   
50         [{'epochs': 5, 'learning_rate': 0.001, 'layers'... True            original                           0.4633 ± 0.0200   
           {'epochs': 2, '

# Reference results

In [54]:
paper_results = """
window_len,train_config,multiple_digits,depth_channel_representation_mode,Accuracy,IoU,AP
10,2,False,original,93.33,14.19,13.4
10,5,False,original,95.04,41.05,32.8
10,15,False,original,95.64,55.47,42.3
20,2,False,original,94.23,20.48,18.7
20,5,False,original,95.63,47.24,37.1
20,15,False,original,96.29,58.01,43.2
50,2,False,original,94.76,27.82,23.7
50,5,False,original,95.27,41.73,35.2
50,15,False,original,96.51,60.29,44.6
"""
import io
paper_df = pd.read_csv(io.StringIO(paper_results))
paper_df = paper_df.groupby(['window_len','train_config','multiple_digits','depth_channel_representation_mode']).mean() / 100
paper_df.head()

Accuracy  \
window_len train_config multiple_digits depth_channel_representation_mode             
10         2            False           original                             0.9333   
           5            False           original                             0.9504   
           15           False           original                             0.9564   
20         2            False           original                             0.9423   
           5            False           original                             0.9563   

                                                                              IoU  \
window_len train_config multiple_digits depth_channel_representation_mode           
10         2            False           original                           0.1419   
           5            False           original                           0.4105   
           15           False           original                           0.5547   
20         2            False           original                           0.2048   
           5            False           original                           0.4724   

                                                                              AP  
window_len train_config multiple_digits depth_channel_representation_mode         
10         2            False           original                           0.134  
           5            False           original                           0.328  
           15           False           original                           0.423  
20         2            False           original                           0.187  
           5            False           original                           0.371

In [55]:
# Load reproduction results
reproduction_results = """
window_len,train_config,multiple_digits,depth_channel_representation_mode,Accuracy,Accuracy_std,IoU,IoU_std,AP,AP_std
10,2,False,original,93.37,0.71,21.08,3.31,19.92,2.34
10,5,False,original,94.93,0.29,38.55,3.30,31.79,2.09
10,15,False,original,96.06,0.11,55.21,2.05,40.23,1.05
20,2,False,original,93.16,1.18,19.18,4.86,18.65,3.26
20,5,False,original,95.25,0.18,42.46,2.27,33.99,1.40
20,15,False,original,96.18,0.04,55.93,1.22,41.53,0.76
50,2,False,original,90.39,3.63,17.01,7.61,18.12,6.83
50,5,False,original,93.10,3.97,35.63,14.81,29.22,12.22
50,15,False,original,96.14,0.05,55.23,1.05,42.25,0.76
"""
import io
reproduction_df = pd.read_csv(io.StringIO(reproduction_results))

# Create MultiIndex
reproduction_metrics = ['Accuracy', 'IoU', 'AP']
reproduction_stds = ['Accuracy_std', 'IoU_std', 'AP_std']

# Set up MultiIndex like paper_df
reproduction_df = reproduction_df.set_index(['window_len', 'train_config', 'multiple_digits', 'depth_channel_representation_mode'])

# Convert percentages to proportions (0-1 scale)
for metric in reproduction_metrics:
    reproduction_df[metric] = reproduction_df[metric] / 100
    reproduction_df[f'{metric}_std'] = reproduction_df[f'{metric}_std'] / 100

# T-Test

In [56]:
# Prepare for t-test and make sure indices match no matter the representation mode
print("Paper index:", paper_df.index)
print("Our means index:", means.index)

# Modify your results to match paper's representation mode for comparison
modified_means = means.copy()
modified_stds = stds.copy()

# Create new indices with 'original' representation mode
modified_index = []
for idx in means.index:
    # Replace the representation mode with 'original'
    modified_idx = (idx[0], idx[1], idx[2], 'original')
    modified_index.append(modified_idx)

# Apply new indices
modified_means.index = pd.MultiIndex.from_tuples(
    modified_index,
    names=['window_len', 'train_config', 'multiple_digits', 'depth_channel_representation_mode']
)
modified_stds.index = modified_means.index

# Use these modified DataFrames in your t-test
common_index = paper_df.index.intersection(modified_means.index)
print(f"Common index length after modification: {len(common_index)}")

Paper index: MultiIndex([(10,  2, False, 'original'),
            (10,  5, False, 'original'),
            (10, 15, False, 'original'),
            (20,  2, False, 'original'),
            (20,  5, False, 'original'),
            (20, 15, False, 'original'),
            (50,  2, False, 'original'),
            (50,  5, False, 'original'),
            (50, 15, False, 'original')],
           names=['window_len', 'train_config', 'multiple_digits', 'depth_channel_representation_mode'])
Our means index: MultiIndex([(10,  2, False, 'zeros'),
            (10,  5, False, 'zeros'),
            (10, 15, False, 'zeros'),
            (20,  2, False, 'zeros'),
            (20,  5, False, 'zeros'),
            (20, 15, False, 'zeros'),
            (50,  2, False, 'zeros'),
            (50,  5, False, 'zeros'),
            (50, 15, False, 'zeros')],
           names=['window_len', 'train_config', 'multiple_digits', 'depth_channel_representation_mode'])
Common index length after modification: 9


In [57]:
from scipy import stats
import numpy as np

# Common setup for both test types
metrics = ['Accuracy', 'IoU', 'AP']
t_test_results = {}

# Print appropriate header based on comparison type
if COMPARE_RESULTS_WITH == 'reproduced_results':
    print("T-test Results:")
    print("H0: Our results = Reproduction results")
    print("H1: Our results ≠ Reproduction results")
else:
    print("T-test Results:")
    print("H0: Our results = Paper results")
    print("H1: Our results ≠ Paper results")

print("\nSignificance levels: * p<0.05, ** p<0.01, *** p<0.001")
print("\nResults (t-statistic, p-value):")

# Loop through metrics once (common to both test types)
for metric in metrics:
    print(f"\n--- {metric} ---")
    t_test_results[metric] = {}
    
    for idx in common_index:
        # Get our sample data (common to both test types)
        our_mean = modified_means.loc[idx, metric]
        our_std = modified_stds.loc[idx, f'{metric}_std']
        
        # Calculate test statistic based on comparison type
        if COMPARE_RESULTS_WITH == 'reproduced_results':
            # Two-sample t-test (Welch's t-test for unequal variances)
            our_n = 5  # Number of runs in our experiment
            repro_mean = reproduction_df.loc[idx, metric]
            repro_std = reproduction_df.loc[idx, f'{metric}_std'] 
            repro_n = 6  # Number of runs in reproduction
            
            # Calculate Welch's t-test
            numerator = our_mean - repro_mean
            denominator = np.sqrt((our_std**2/our_n) + (repro_std**2/repro_n))
            t_stat = numerator / denominator
            
            # Calculate degrees of freedom for Welch's t-test
            df_numerator = (our_std**2/our_n + repro_std**2/repro_n)**2
            df_denominator = (our_std**4/(our_n**2 * (our_n-1))) + (repro_std**4/(repro_n**2 * (repro_n-1)))
            df = df_numerator / df_denominator
            
            # Calculate p-value (two-tailed test)
            p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=df))
            
            # Store results
            t_test_results[metric][idx] = {
                't_statistic': t_stat,
                'p_value': p_value,
                'our_mean': our_mean,
                'our_std': our_std,
                'repro_mean': repro_mean,
                'repro_std': repro_std,
                'df': df
            }
        else:
            # One-sample t-test against paper values
            paper_mean = paper_df.loc[idx, metric]
            n = 6  # Sample size
            
            # Calculate t-statistic for 1-sample t-test
            t_stat = (our_mean - paper_mean) / (our_std / np.sqrt(n))
            
            # Calculate p-value (two-tailed test)
            p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))
            
            # Store results
            t_test_results[metric][idx] = {
                't_statistic': t_stat,
                'p_value': p_value,
                'our_mean': our_mean,
                'paper_mean': paper_mean,
                'our_std': our_std
            }
        
        # Determine significance level (common code)
        if p_value < 0.001:
            sig = "***"
        elif p_value < 0.01:
            sig = "**"
        elif p_value < 0.05:
            sig = "*"
        else:
            sig = ""
        
        # Print results based on comparison type
        if COMPARE_RESULTS_WITH == 'reproduced_results':
            print(f"{idx}: t={t_stat:.3f}, p={p_value:.3f}{sig}, df={df:.1f}")
            print(f"  Our: {our_mean*100:.2f}±{our_std*100:.2f}% vs Repro: {repro_mean*100:.2f}±{repro_std*100:.2f}%")
        else:
            print(f"{idx}: t={t_stat:.3f}, p={p_value:.3f}{sig}")
            print(f"  Our: {our_mean*100:.2f}±{our_std*100:.2f}% vs Paper: {paper_mean*100:.2f}%")


print("\n" + "="*50)
print("SUMMARY")
print("="*50)

for metric in metrics:
    p_values = [result['p_value'] for result in t_test_results[metric].values()]
    significant_count = sum(1 for p in p_values if p < 0.05)
    
    print(f"\n{metric}:")
    print(f"  Total comparisons: {len(p_values)}")
    print(f"  Significant differences (p<0.05): {significant_count}")
    print(f"  Non-significant: {len(p_values) - significant_count}")

T-test Results:
H0: Our results = Reproduction results
H1: Our results ≠ Reproduction results

Significance levels: * p<0.05, ** p<0.01, *** p<0.001

Results (t-statistic, p-value):

--- Accuracy ---
(10, 2, False, 'original'): t=0.996, p=0.346, df=8.8
  Our: 93.79±0.67% vs Repro: 93.37±0.71%
(10, 5, False, 'original'): t=0.571, p=0.585, df=7.4
  Our: 95.05±0.38% vs Repro: 94.93±0.29%
(10, 15, False, 'original'): t=0.021, p=0.984, df=5.8
  Our: 96.06±0.21% vs Repro: 96.06±0.11%
(20, 2, False, 'original'): t=-0.784, p=0.453, df=9.0
  Our: 92.64±1.03% vs Repro: 93.16±1.18%
(20, 5, False, 'original'): t=-0.222, p=0.830, df=7.0
  Our: 95.22±0.25% vs Repro: 95.25±0.18%
(20, 15, False, 'original'): t=-2.439, p=0.058, df=5.1
  Our: 96.06±0.10% vs Repro: 96.18±0.04%
(50, 2, False, 'original'): t=2.223, p=0.071, df=5.6
  Our: 93.78±0.81% vs Repro: 90.39±3.63%
(50, 5, False, 'original'): t=1.151, p=0.299, df=5.3
  Our: 94.99±0.63% vs Repro: 93.10±3.97%
(50, 15, False, 'original'): t=1.145, p=0.2

# Holm-Bonferroni

In [58]:
from statsmodels.stats.multitest import multipletests
import pandas as pd

# Collect all p-values and their corresponding information
all_tests = []
for metric in metrics:
    for idx, result in t_test_results[metric].items():
        test_data = {
            'metric': metric,
            'configuration': idx,
            'p_value': result['p_value'],
            't_statistic': result['t_statistic'],
            'our_mean': result['our_mean'],
            'our_std': result['our_std']
        }
        
        # Add reference values based on comparison type
        if COMPARE_RESULTS_WITH == 'reproduced_results':
            test_data.update({
                'repro_mean': result['repro_mean'],
                'repro_std': result['repro_std'],
                'df': result['df']
            })
        else:
            test_data.update({
                'paper_mean': result['paper_mean']
            })
        
        all_tests.append(test_data)

# Extract p-values for correction
p_values = [test['p_value'] for test in all_tests]

# Apply Holm-Bonferroni correction
rejected, p_corrected, alpha_sidak, alpha_bonf = multipletests(
    p_values, 
    alpha=0.05, 
    method='holm'
)

# Add corrected results back to our test data
for i, test in enumerate(all_tests):
    test['p_corrected'] = p_corrected[i]
    test['rejected_holm'] = rejected[i]

# Set correct hypothesis text based on comparison type
hypothesis_text = "Reproduction results" if COMPARE_RESULTS_WITH == 'reproduced_results' else "Paper results"

print("HOLM-BONFERRONI CORRECTED RESULTS")
print("="*60)
print(f"H0: Our results = {hypothesis_text}")
print(f"H1: Our results ≠ {hypothesis_text}")
print(f"Total number of tests: {len(p_values)}")
print(f"Family-wise error rate (FWER) controlled at α = 0.05")
print("\nSignificance levels: * p_corrected<0.05, ** p_corrected<0.01, *** p_corrected<0.001")
print("\nResults (t-statistic, p_original → p_corrected, significant):")

# Display results organized by metric
for metric in metrics:
    print(f"\n--- {metric} ---")
    
    metric_tests = [test for test in all_tests if test['metric'] == metric]
    
    for test in metric_tests:
        # Determine significance level for corrected p-value
        if test['p_corrected'] < 0.001:
            sig = "***"
        elif test['p_corrected'] < 0.01:
            sig = "**"
        elif test['p_corrected'] < 0.05:
            sig = "*"
        else:
            sig = ""
        
        # Show if hypothesis was rejected
        rejected_status = "SIGNIFICANT" if test['rejected_holm'] else "not significant"
        
        print(f"{test['configuration']}:")
        if COMPARE_RESULTS_WITH == 'reproduced_results':
            print(f"  t={test['t_statistic']:.3f}, p={test['p_value']:.3f} → p_corrected={test['p_corrected']:.3f}{sig}, df={test['df']:.1f}")
            print(f"  Our: {test['our_mean']*100:.2f}±{test['our_std']*100:.2f}% vs Repro: {test['repro_mean']*100:.2f}±{test['repro_std']*100:.2f}%")
        else:
            print(f"  t={test['t_statistic']:.3f}, p={test['p_value']:.3f} → p_corrected={test['p_corrected']:.3f}{sig}")
            print(f"  Our: {test['our_mean']*100:.2f}±{test['our_std']*100:.2f}% vs Paper: {test['paper_mean']*100:.2f}%")
        print(f"  Result: {rejected_status}")

# Summary statistics
print("\n" + "="*60)
print("SUMMARY - HOLM-BONFERRONI CORRECTED")
print("="*60)

total_significant_original = sum(1 for test in all_tests if test['p_value'] < 0.05)
total_significant_corrected = sum(1 for test in all_tests if test['rejected_holm'])

print(f"Total tests performed: {len(all_tests)}")
print(f"Significant before correction (p<0.05): {total_significant_original}")
print(f"Significant after Holm-Bonferroni correction: {total_significant_corrected}")
print(f"Reduction due to multiple comparisons correction: {total_significant_original - total_significant_corrected}")

# Summary by metric
for metric in metrics:
    metric_tests = [test for test in all_tests if test['metric'] == metric]
    original_sig = sum(1 for test in metric_tests if test['p_value'] < 0.05)
    corrected_sig = sum(1 for test in metric_tests if test['rejected_holm'])
    
    print(f"\n{metric}:")
    print(f"  Total comparisons: {len(metric_tests)}")
    print(f"  Significant before correction: {original_sig}")
    print(f"  Significant after correction: {corrected_sig}")
    print(f"  Proportion significant after correction: {corrected_sig/len(metric_tests):.1%}")

# Create a summary DataFrame for easy viewing
summary_df = pd.DataFrame(all_tests)
summary_df['configuration_str'] = summary_df['configuration'].astype(str)
summary_df['significant_original'] = summary_df['p_value'] < 0.05
summary_df['significant_corrected'] = summary_df['rejected_holm']

print(f"\nDetailed results saved in 'summary_df' DataFrame with columns:")
print(f"  {list(summary_df.columns)}")

# Save detailed results to CSV
summary_df.to_csv('holm_bonferroni_results.csv', index=False)
print(f"\nDetailed results saved to 'holm_bonferroni_results.csv'")


HOLM-BONFERRONI CORRECTED RESULTS
H0: Our results = Reproduction results
H1: Our results ≠ Reproduction results
Total number of tests: 27
Family-wise error rate (FWER) controlled at α = 0.05

Significance levels: * p_corrected<0.05, ** p_corrected<0.01, *** p_corrected<0.001

Results (t-statistic, p_original → p_corrected, significant):

--- Accuracy ---
(10, 2, False, 'original'):
  t=0.996, p=0.346 → p_corrected=1.000, df=8.8
  Our: 93.79±0.67% vs Repro: 93.37±0.71%
  Result: not significant
(10, 5, False, 'original'):
  t=0.571, p=0.585 → p_corrected=1.000, df=7.4
  Our: 95.05±0.38% vs Repro: 94.93±0.29%
  Result: not significant
(10, 15, False, 'original'):
  t=0.021, p=0.984 → p_corrected=1.000, df=5.8
  Our: 96.06±0.21% vs Repro: 96.06±0.11%
  Result: not significant
(20, 2, False, 'original'):
  t=-0.784, p=0.453 → p_corrected=1.000, df=9.0
  Our: 92.64±1.03% vs Repro: 93.16±1.18%
  Result: not significant
(20, 5, False, 'original'):
  t=-0.222, p=0.830 → p_corrected=1.000, df=7

# Results Tables

In [59]:
# Reshape results back to original table format
def create_results_table(all_tests, value_column, metrics):
    """Helper function to create a table with original indexing"""
    results_dict = {}
    
    for test in all_tests:
        config = test['configuration']
        metric = test['metric']
        value = test[value_column]
        
        if config not in results_dict:
            results_dict[config] = {}
        results_dict[config][metric] = value
    
    # Convert to DataFrame with proper indexing
    results_df = pd.DataFrame.from_dict(results_dict, orient='index')
    results_df = results_df.reindex(columns=metrics)  # Ensure column order
    return results_df

# Create tables for different aspects of the results
metrics = ['Accuracy', 'IoU', 'AP']

# Table 1: Original p-values
p_values_table = create_results_table(all_tests, 'p_value', metrics)

# Table 2: Holm-Bonferroni corrected p-values
p_corrected_table = create_results_table(all_tests, 'p_corrected', metrics)

# Table 3: T-statistics
t_stats_table = create_results_table(all_tests, 't_statistic', metrics)

# Table 4: Significance status (True = significant after correction)
significance_table = create_results_table(all_tests, 'rejected_holm', metrics)

print("ORIGINAL P-VALUES")
print("="*50)
print(p_values_table.round(4))

print("\n\nHOLM-BONFERRONI CORRECTED P-VALUES")
print("="*50)
print(p_corrected_table.round(4))

print("\n\nT-STATISTICS")
print("="*50)
print(t_stats_table.round(3))

print("\n\nSIGNIFICANCE STATUS (after Holm-Bonferroni correction)")
print("="*50)
print("True = Significant difference from paper results")
print(significance_table)

# Create a combined summary table showing key information
print("\n\nCOMBINED SUMMARY TABLE")
print("="*50)
combined_summary = pd.DataFrame(index=p_values_table.index)

for metric in metrics:
    # Format: "p_orig → p_corr (sig_status)"
    combined_summary[f'{metric}_p_original'] = p_values_table[metric].round(4)
    combined_summary[f'{metric}_p_corrected'] = p_corrected_table[metric].round(4)
    combined_summary[f'{metric}_significant'] = significance_table[metric]
    
    # Create a readable summary column
    combined_summary[f'{metric}_summary'] = combined_summary.apply(
        lambda row: f"{row[f'{metric}_p_original']:.3f} → {row[f'{metric}_p_corrected']:.3f} {'*' if row[f'{metric}_significant'] else ''}", 
        axis=1
    )

# Show just the summary columns for readability
summary_columns = [f'{metric}_summary' for metric in metrics]
print(combined_summary[summary_columns])

# Create a table showing differences from paper (Our - Paper)
print("\n\nDIFFERENCES FROM PAPER RESULTS (Our - Paper)")
print("="*50)


# If comparing with reproduction results
if COMPARE_RESULTS_WITH == 'reproduced_results':
    differences_table = pd.DataFrame(index=reproduction_df.index)

    for metric in metrics:
        # Calculate differences (already on same scale)
        our_values = modified_means.loc[reproduction_df.index, metric] * 100
        repro_values = reproduction_df.loc[reproduction_df.index, metric] * 100
        differences_table[metric] = our_values - repro_values

    print(differences_table.round(2))

# If comparing with original paper results
else:
    differences_table = pd.DataFrame(index=paper_df.index)

    for metric in metrics:
        # Calculate differences (convert means back to same scale as paper)
        our_values = modified_means.loc[paper_df.index, metric] * 100  # Convert to percentage
        paper_values = paper_df[metric]
        differences_table[metric] = our_values - paper_values

    print(differences_table.round(2))

# Mark significant differences with asterisks
print("\n\nDIFFERENCES WITH SIGNIFICANCE MARKERS")
print("="*50)
print("* = Significant difference after Holm-Bonferroni correction")

differences_with_sig = differences_table.copy()
for metric in metrics:
    differences_with_sig[metric] = differences_with_sig[metric].round(2).astype(str)
    
    # Add asterisks for significant differences
    sig_mask = significance_table[metric]
    differences_with_sig.loc[sig_mask, metric] = differences_with_sig.loc[sig_mask, metric] + '*'

print(differences_with_sig)

# Save all tables
p_values_table.to_csv('p_values_original.csv')
p_corrected_table.to_csv('p_values_holm_corrected.csv')
t_stats_table.to_csv('t_statistics.csv')
significance_table.to_csv('significance_status.csv')
differences_with_sig.to_csv('differences_from_paper.csv')

print(f"\n\nAll tables saved as CSV files:")
print("- p_values_original.csv")
print("- p_values_holm_corrected.csv") 
print("- t_statistics.csv")
print("- significance_status.csv")
print("- differences_from_paper.csv")

ORIGINAL P-VALUES
                      Accuracy     IoU      AP
10 2  False original    0.3456  0.9305  0.8288
   5  False original    0.5852  0.5406  0.3639
   15 False original    0.9842  0.8425  0.8061
20 2  False original    0.4532  0.2633  0.7617
   5  False original    0.8304  0.6046  0.9642
   15 False original    0.0582  0.2646  0.5116
50 2  False original    0.0712  0.4966  0.6566
   5  False original    0.2991  0.4707  0.3782
   15 False original    0.2818  0.8003  0.1135


HOLM-BONFERRONI CORRECTED P-VALUES
                      Accuracy  IoU   AP
10 2  False original       1.0  1.0  1.0
   5  False original       1.0  1.0  1.0
   15 False original       1.0  1.0  1.0
20 2  False original       1.0  1.0  1.0
   5  False original       1.0  1.0  1.0
   15 False original       1.0  1.0  1.0
50 2  False original       1.0  1.0  1.0
   5  False original       1.0  1.0  1.0
   15 False original       1.0  1.0  1.0


T-STATISTICS
                      Accuracy    IoU     AP
10 2 